# Lekcija 37: Klasične arhitekture

U lekcijama 34–35 izgradili smo i trenirali jedan mali CNN. Ova lekcija razmatra razvoj *arhitektura* CNN-ova tijekom približno dvaju desetljeća i detaljno obrađuje najvažniju arhitekturnu ideju iz tog razdoblja: rezidualnu vezu i problem nestajanja gradijenta za čije je rješavanje osmišljena.

## ImageNet i ILSVRC: referentni zadatak koji je pokretao razvoj

**ImageNet** (<a href="../references.html#deng-2009">Deng i sur., 2009.</a><span class="landmark-paper">&#9733;</span>) je *skup podataka*: više od 14 milijuna slika ručno označenih u više od 20 000 kategorija, izgrađen masovnim prikupljanjem oznaka prema postojećoj hijerarhiji imenica u WordNetu. No sam veliki označeni skup podataka ne pokreće napredak &mdash; zajedničko, standardizirano *natjecanje* to čini. Od 2010. **ILSVRC** (ImageNet Large Scale Visual Recognition Challenge, natjecanje u vizualnom prepoznavanju velikog opsega na ImageNetu, <a href="../references.html#russakovsky-2015">Russakovsky i sur., 2015.</a><span class="landmark-paper">&#9733;</span>) pretvorio je podskup ImageNeta s 1000 kategorija i oko 1.2 milijuna slika u godišnji referentni zadatak: svaka je skupina trenirala na istim podacima i ocjenjivala se na istom izdvojenom testnom skupu, čime su arhitekture postale izravno i objektivno usporedive u dotad nedostignutom opsegu. Svaka arhitektura u pregledu u nastavku prijavljuje rezultat kao sudionik ILSVRC-a, a stopa pogreške pobjednika među prvih pet predviđanja po godinama dobar je pokazatelj napretka područja tijekom desetljeća koje lekcija obrađuje:

| Godina | Pobjednik | Pogreška među prvih 5 predviđanja |
|---|---|---|
| 2010 | (značajke prije dubokog učenja) | 28.2% |
| 2012 | AlexNet | 16.4% |
| 2014 | GoogLeNet (VGG blizu iza njega, sa 7.3%) | 6.7% |
| 2015 | ResNet | 3.57% |
| 2017 | SENet | 2.25% |

Na istom zadatku uvježban čovjek postiže stopu pogreške od oko 5 %. ResNet je prvi nadmašio taj rezultat &mdash; što se ponekad naziva „nadljudskom” uspješnošću. Do 2017., uz pobjedničku pogrešku ispod 2.5 %, klasifikacijska disciplina ILSVRC-a završila je &mdash; njezina uloga referentnog zadatka bila je uglavnom ispunjena.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## Kratak povijesni pregled

Tijekom tog razdoblja nastalo je nekoliko arhitektura, od kojih je svaka rješavala određen problem:

- **LeNet-5** („Učenje temeljeno na gradijentu primijenjeno na prepoznavanje dokumenata”, <a href="../references.html#lecun-1998">LeCun i sur., 1998.</a><span class="landmark-paper">&#9733;</span>) — obrazac već implementiran u lekciji 34: konvolucija, sažimanje, konvolucija, sažimanje, zatim potpuno povezani slojevi. Izgrađen za prepoznavanje znamenki na slikama 32x32.
- **AlexNet** (<a href="../references.html#krizhevsky-2012">Krizhevsky, Sutskever & Hinton, 2012.</a><span class="landmark-paper">&#9733;</span>) — isti obrazac, samo mnogo veći (5 konvolucijskih slojeva, oko 60 milijuna parametara), treniran na ImageNetu pomoću GPU-a i ReLU-a umjesto tanh/sigmoid aktivacija. Njegova pobjeda na ILSVRC-u 2012. (vidjeti gornju tablicu) prekretnica je koja je pokrenula revoluciju dubokog učenja u računalnom vidu &mdash; obnavljajući širi interes za neuronske mreže nakon desetljeća zastoja (sličnog „zimi umjetne inteligencije”).
- **VGG** (<a href="../references.html#simonyan-zisserman-2014">Simonyan & Zisserman, 2014.</a><span class="landmark-paper">&#9733;</span>) — zamijenio je AlexNetove velike filtre 11x11/5x5 dubokim nizom malih konvolucija 3x3. Dvije uzastopne konvolucije 3x3 imaju isto *receptivno polje* kao jedna konvolucija 5x5 (ponovno ideja piramida iz lekcije 11), ali manje parametara i dodatnu nelinearnost između njih.
- **GoogLeNet / Inception** (<a href="../references.html#szegedy-2015">Szegedy i sur., 2015.</a><span class="landmark-paper">&#9733;</span>) — proširivao je mrežu umjesto da je samo produbljuje: svaki „Inception modul” usporedno primjenjuje više veličina konvolucije (1x1, 3x3, 5x5) na isti ulaz i spaja njihove izlaze, pa se mreža ne mora ograničiti na jednu veličinu filtra po sloju. Najprije se koriste konvolucije 1x1 za jeftino smanjivanje broja kanala, čime širi modul ostaje računalno prihvatljiv.
- **ResNet** (<a href="../references.html#he-2015-resnet">He i sur., 2015.</a><span class="landmark-paper">&#9733;</span>) — povećao je dubinu s VGG-ovih oko 19 slojeva na 50, 101, pa čak i 152, rješavanjem problema zbog kojeg je naivno dodavanje slojeva davalo *lošije*, a ne bolje rezultate.
- **SENet** (<a href="../references.html#hu-2018-senet">Hu, Shen & Sun, 2018.</a><span class="landmark-paper">&#9733;</span>), pobjednik ILSVRC-a 2017. iz gornje tablice — nije novi oblik osnovne mreže, nego usavršavanje: blok „squeeze-and-excitation” omogućuje svakom sloju da nauči težine po kanalima (koji su kanali značajki najvažniji za ovaj ulaz), umjesto da sve kanale tretira jednako.

Ostatak lekcije usredotočuje se na ResNetovu rezidualnu vezu kao najutjecajniju pojedinačnu ideju iz ove povijesti.

## Izgradnja ResNet bloka

### Problem nestajanja gradijenta

Zašto dodavanje slojeva obične mreže čini *lošijima*? Lančano pravilo povratnog širenja pogreške množi gradijent lokalnim Jacobianom svakog sloja na putu natrag prema ulazu. Ako su ti faktori po slojevima dosljedno manji od 1 — što se lako događa sa zasićujućom aktivacijom poput sigmoidne funkcije, čija je derivacija najviše 0.25 — dovoljno dubok niz mnogo puta množi gradijent vrlo malim brojem. Gradijent koji stiže do ranih slojeva smanjuje se prema nuli i ti slojevi potpuno prestaju učiti.

Izgradite mrežu od 30 slojeva `Linear` sa sigmoidnom aktivacijom i pratite veličinu gradijenta na svakoj dubini za dvije varijante: **obični** niz `x = sigmoid(layer(x))` i **rezidualni** niz `x = x + 0.3 * sigmoid(layer(x))`, u kojem svaki sloj treba naučiti samo malu *korekciju* koja se dodaje njegovu ulazu umjesto da ga potpuno zamijeni.

In [ ]:
DEPTH = 30
WIDTH = 32

def make_layers(seed):
    torch.manual_seed(seed)
    layers = nn.ModuleList([nn.Linear(WIDTH, WIDTH) for _ in range(DEPTH)])
    for layer in layers:
        nn.init.xavier_normal_(layer.weight, gain=nn.init.calculate_gain('sigmoid'))
    return layers

class PlainDeepNet(nn.Module):
    def __init__(self, seed):
        super().__init__()
        self.layers = make_layers(seed)

    def forward(self, x):
        acts = [x]
        for layer in self.layers:
            x = torch.sigmoid(layer(x))
            x.retain_grad()
            acts.append(x)
        return x, acts

class ResidualDeepNet(nn.Module):
    def __init__(self, seed):
        super().__init__()
        self.layers = make_layers(seed)

    def forward(self, x):
        acts = [x]
        for layer in self.layers:
            x = x + 0.3 * torch.sigmoid(layer(x))
            x.retain_grad()
            acts.append(x)
        return x, acts

def grad_norms(model_cls):
    torch.manual_seed(0)
    x = torch.randn(8, WIDTH, requires_grad=True)
    model = model_cls(seed=1)
    out, acts = model(x)
    out.sum().backward()
    return [a.grad.norm().item() for a in acts if a.grad is not None]

plain_norms = grad_norms(PlainDeepNet)
res_norms = grad_norms(ResidualDeepNet)

print(f'plain net:    gradient norm at layer 0 / gradient norm at layer {DEPTH-1} = {plain_norms[0] / plain_norms[-1]:.2e}')
print(f'residual net: gradient norm at layer 0 / gradient norm at layer {DEPTH-1} = {res_norms[0] / res_norms[-1]:.2e}')

plt.figure(figsize=(6, 4))
plt.semilogy(plain_norms, label='plain (sigmoid) net')
plt.semilogy(res_norms, label='residual net')
plt.xlabel('layer depth (0 = input)')
plt.ylabel('gradient norm (log scale)')
plt.title('Gradient magnitude vs. depth')
plt.legend()
plt.show()

Gradijent obične mreže smanjuje se za približno devetnaest redova veličine između posljednjeg i prvog sloja — za sve praktične potrebe rani slojevi uopće ne dobivaju signal za učenje. Gradijent rezidualne mreže (ResNeta) ostaje gotovo stalan kroz svih 30 slojeva.

Razlog je struktura, a ne podešavanje: uz rezidualnu vezu vrijedi $x_{l+1} = x_l + F(x_l)$, pa je $dx_{l+1}/dx_l = I + dF/dx_l$. Povratno širenje pogreške množi te Jacobiane kroz slojeve, ali svaki od njih kao izravan član sadrži jediničnu matricu $I$. Time gradijent dobiva put ravno do ulaza kojim se nikad ne množi malom derivacijom sigmoidne funkcije — neometan prečac, neovisno o dubini niza. Obična mreža nema takav put: Jacobian svakog sloja samo je $dF/dx_l$, pa ništa ne sprječava uzastopno množenje da umnožak svede prema nuli.

### Normalizacija serije

ResNet koristi još jedan sastojak: **normalizaciju serije (batch normalization)** (<a href="../references.html#ioffe-szegedy-2015">Ioffe & Szegedy, 2015.</a><span class="landmark-paper">&#9733;</span>). Ideja je jednostavna &mdash; za svaki kanal oduzmite njegovu srednju vrijednost i podijelite njegovom standardnom devijacijom, izračunatima *preko trenutačne mini-serije* (po dimenzijama serije, visine i širine, zasebno za svaki kanal), kako bi aktivacije svakog kanala pri ulasku u sljedeći sloj imale srednju vrijednost 0 i standardnu devijaciju 1. Zatim se primjenjuju dva parametra koja se mogu učiti, faktor skaliranja `gamma` i pomak `beta` po kanalu, pa sloj ipak može uspostaviti drukčiju srednju vrijednost ili mjerilo ako je to korisno &mdash; normalizacija na 0/1 *polazište* je koje mreža može poništiti, a ne strogo ograničenje.

Duboke mreže bez normalizacije serije sklone su problemu povezanom s nestajanjem gradijenata, ali različitom od njega: kako treniranje ažurira rane slojeve, *raspodjela* aktivacija koje ulaze u kasnije slojeve stalno se mijenja (ponekad se naziva „unutarnji pomak kovarijata”), pa kasniji slojevi stalno prate pomičnu metu. Ponovna normalizacija u svakom sloju održava tu raspodjelu stabilnom, što u praksi dopušta mnogo veće stope učenja i primjetno olakšava treniranje dubokih mreža &mdash; jedan je to od razloga zbog kojih je ResNet mogao doseći više od 50 slojeva, dok su ranije arhitekture imale poteškoća iznad približno 20.

In [ ]:
C = 4
x_bn = torch.randn(8, C, 5, 5)

bn = nn.BatchNorm2d(C)
bn.train()
out_torch = bn(x_bn)

# from scratch: normalize each channel over (batch, height, width), then scale + shift
mean = x_bn.mean(dim=(0, 2, 3), keepdim=True)
var = x_bn.var(dim=(0, 2, 3), unbiased=False, keepdim=True)
x_norm = (x_bn - mean) / torch.sqrt(var + bn.eps)
gamma = bn.weight.view(1, C, 1, 1)
beta = bn.bias.view(1, C, 1, 1)
out_manual = gamma * x_norm + beta

print(f'max abs diff vs nn.BatchNorm2d: {(out_manual - out_torch).abs().max().item():.2e}')
print(f'per-channel mean before: {x_bn.mean(dim=(0, 2, 3)).detach().numpy().round(3)}')
print(f'per-channel std before:  {x_bn.std(dim=(0, 2, 3), unbiased=False).detach().numpy().round(3)}')
print(f'per-channel mean after:  {out_torch.mean(dim=(0, 2, 3)).detach().numpy().round(3)}')
print(f'per-channel std after:   {out_torch.std(dim=(0, 2, 3), unbiased=False).detach().numpy().round(3)}')

### Povezivanje svega: konvolucijski rezidualni blok

U stvarnom ResNetu `F` je par malih konvolucija (s normalizacijom serije i ReLU-om između njih), a prečac — koji se naziva i **preskočna veza**, jer potpuno preskače `F` — dodaje *ulaznu kartu značajki* natrag njihovom izlazu, kanal po kanal i piksel po piksel:

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(x + out)  # the shortcut: add the block's input back in

block = ResidualBlock(channels=16)
x = torch.randn(4, 16, 20, 20)
y = block(x)
print(f'input shape:  {tuple(x.shape)}')
print(f'output shape: {tuple(y.shape)}  (unchanged — a residual block preserves shape, so blocks can be stacked freely)')

### Zadaci

1. U gornjim `PlainDeepNet`/`ResidualDeepNet` promijenite aktivaciju s `torch.sigmoid` na `torch.relu` (ReLU-ova derivacija je 1 za svaki pozitivan ulaz, a ne ograničena na 0.25 kao sigmoidna) i ponovite usporedbu normi gradijenta. Ublažava li se problem nestajanja u običnoj mreži, pogoršava li se ili ostaje približno isti?
2. Promijenite faktor skaliranja rezidualne mreže s `0.3` na `1.0` (odnosno `x + torch.sigmoid(layer(x))` bez prigušenja) i ponovno pokrenite. Mijenja li se omjer gradijenata znatno? Što to sugerira o *razlogu* djelovanja rezidualne veze — je li presudan faktor skaliranja ili sam prečac `+ x`?
3. Gornji `ResidualBlock` zahtijeva isti broj kanala na ulazu i izlazu jer se izravno zbrajaju. Stvarni ResNetovi ponekad trebaju promijeniti broj kanala između blokova (npr. 16 → 32). Opišite riječima ili kodom što bi put prečaca tada morao učiniti da zbrajanje i dalje ima smisla.